# 02 LeRobot训练流程与策略学习

目标：从数据结构进入训练流程。

补充内容：transition、stats归一化、时间窗口、DataLoader、batch、ACT/Diffusion Policy。

In [1]:
import sys
import torch
print(sys.executable)
print(torch.__version__)
print('CUDA:', torch.cuda.is_available())

d:\Desktop\robot\envs\lerobot-win\python.exe
2.11.0+cu128
CUDA: True


## 1. 数据、episode、frame、transition

episode是一段完整任务轨迹。
frame是一个采样时刻。
transition表示：observation_t + action_t -> observation_t+1。

数据集提供采样结果，不提供物理转移函数f(s,a)。

In [2]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset('lerobot/pusht')
sample = dataset[0]
print(sample.keys())
print(sample['observation.state'])
print(sample['action'])

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 667.30it/s]


dict_keys(['observation.image', 'observation.state', 'action', 'episode_index', 'frame_index', 'timestamp', 'next.reward', 'next.done', 'next.success', 'index', 'task_index', 'task'])
tensor([222.,  97.])
tensor([233.,  71.])


## 2. metadata.stats与归一化

不同传感器尺度不同，需要归一化。

常见：x_norm=(x-mean)/std

In [3]:
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

metadata = LeRobotDatasetMetadata('lerobot/pusht')
print(metadata.stats.keys())

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 666.64it/s]

dict_keys(['index', 'next.success', 'observation.state', 'next.done', 'observation.image', 'timestamp', 'episode_index', 'frame_index', 'action', 'task_index', 'next.reward'])


## 3. 时间窗口与action chunk

ACT和Diffusion Policy通常使用连续动作序列。

输入：observation_t

输出：action_t:t+N

## 4. DataLoader和batch

Dataset返回一个sample。
DataLoader负责随机采样并组合batch。

单图片：(3,H,W)

batch图片：(B,3,H,W)

In [4]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=4, shuffle=True)
batch = next(iter(loader))
print(batch.keys())

dict_keys(['observation.image', 'observation.state', 'action', 'episode_index', 'frame_index', 'timestamp', 'next.reward', 'next.done', 'next.success', 'index', 'task_index', 'task'])


## 5. Policy训练

模仿学习目标：学习 observation -> action。

ACT：Transformer预测未来动作chunk。

Diffusion Policy：通过扩散生成动作序列。

## 6. 下一阶段

1. MuJoCo env.reset/env.step
2. policy连接仿真
3. ACT结构
4. 小规模训练